# 05 - One question, three domains: planning and parallel dispatch

**Scenario.** A support team lead asks one message that touches **HR policy**, a **product manual** and the
**ticket system**. The Coordinator (high reasoning effort) splits it into independent steps and dispatches them in
parallel with LangGraph `Send`; the specialists run at the same time.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

In [1]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

Coordinator: gpt-5.1 (high) | specialists: gpt-5.1 (low) | judge: gpt-5.5 (pass >= 80)


In [2]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

CRM MCP server at http://localhost:8765/mcp: customer=['get_customer_profile', 'search_customer_profiles'], ticket_read=['get_tickets_by_customer', 'search_tickets'], ticket_write=['register_ticket', 'resolve_ticket']
- `hr_policy` (available): HR policies (leave, travel and expenses, medical insurance, conduct, POSH, WFH, IT security, privacy, attendance, appraisal, whistleblowing, anti-bribery, notice period). Current versions by default; flags superseded values.
- `product_manuals` (available): TV and phone user manuals: specifications, setup, error codes, troubleshooting steps, warranty and safety. Pins the exact model (near-duplicate manuals).
- `customer_support` (available): Find customers by ID, name, email or phone and report their profile: active status, credit limit, contact details, remarks such as products owned. Read-only.
- `ticket_management` (available): Support ticket history and search (by customer, status, priority, product model, text). Proposes new tickets or reso

In [3]:
QUESTION = ("I lead the TV support team. (1) If my engineers work overtime this weekend, at what rate is overtime "
            "paid and is there a cap? (2) What does error LX-90 mean on the Lumenix OLED O65 and how is it fixed? "
            "(3) How many OLED O65 tickets are open or in progress right now?")

result = await service.ask(QUESTION, user_id="lead-sanjay", on_event=print)
result

[   0.0s] start_turn      done
[   0.0s] load_memory     1 long-term facts recalled
[   8.8s] plan            Initial plan for this question. -> s1=hr_policy, s2=product_manuals, s3=ticket_management
[  32.7s] run_step        s3 ticket_management done in 23.9s (confidence 0.93, 4 evidence, 0 proposed actions)
[  42.5s] run_step        s2 product_manuals done in 33.6s (confidence 0.90, 1 evidence, 0 proposed actions)
[  53.6s] run_step        s1 hr_policy done in 44.8s (confidence 0.92, 5 evidence, 0 proposed actions)
[  62.8s] reflect         synthesize: All three subquestions are fully addressed by the existing specialist reports: the overtime policy is clearly summarized
[  80.9s] synthesize      done
[  85.0s] judge           attempt 1: 72.5/100 FAIL (ground 55, rel 90, compl 75, ctx 75)
[  90.9s] plan            RETRY after the quality review failed. Previous answer: -> s4=product_manuals
[ 109.9s] run_step        s4 product_manuals done in 18.9s (confidence 0.97, 2 evidence, 0 pro

**status** `completed` | **thread** `thread-84a3cbda` | **elapsed** 156.0s  
**judge** 93.8/100 | **passed** True | **attempts** 2 | **specialist runs** 4

```text
plan round 1 (Initial plan for this question.): s1:hr_policy, s2:product_manuals, s3:ticket_management
plan round 2 (RETRY after the quality review failed. Previous answer:): s4:product_manuals
timeline:
    s1 hr_policy             0.0s ->   44.8s |#################                       |
    s2 product_manuals       0.0s ->   33.6s |#############                           |
    s3 ticket_management     0.0s ->   23.9s |#########                               |
    s4 product_manuals      82.1s ->  101.1s |                                ####### |
judge attempt 1: 72.5/100 FAIL (ground 55, rel 90, compl 75, ctx 75)
judge attempt 2: 93.8/100 PASS (ground 88, rel 98, compl 96, ctx 96)
```

---

1) Weekend overtime for your engineers  

- Under the current Attendance, Working Hours and Overtime Policy (HR‑POL‑009‑V2.3), **only “overtime‑eligible employees” = non‑managerial L1 staff in support or operations roles** are entitled to overtime pay. [E1]  
- **Managers and all employees in grade L2 and above are not eligible for overtime pay; they receive compensatory off instead under the Leave Policy.** [E4]  
- Corporate and engineering roles are listed under the general 9:30–6:30 shift and are not included in the definition of overtime‑eligible roles. [E1][E2]  

So for your engineers this weekend:  
- **They typically would not receive overtime pay** unless someone is formally classified as an L1 support/operations overtime‑eligible employee. [E1][E4]  
- In that usual case (L2+ engineering roles), extra weekend work is compensated through **compensatory time off**, not overtime pay. [E4]  
- I cannot see your team’s exact job bands, so you would need to confirm any exceptions (e.g., L1 roles mapped as support/operations) with HR or in PeopleHub.

Where an employee *is* overtime‑eligible, the policy is:  
- Overtime = work **beyond 9 hours/day or 48 hours/week**. [E3]  
- **Paid at 2× the ordinary rate of wages.** [E4]  
- **Capped at 50 overtime hours per quarter per employee.** [E4]  
- **No higher or special weekend rate is defined**; Saturday/Sunday are weekly offs but overtime pay is still at 2×. [E1][E4]  
- **Pre‑approval in PeopleHub is required before** the extra hours are worked; paid with the next month’s salary. [E4][E5]  

---

2) LX‑90 on Lumenix OLED O65 – meaning and fix  

For the **Lumenix OLED O65 (model LX‑65O9)**:  
- **LX‑90 is defined as “Pixel refresh interrupted.”** [E11]  
- The official prescribed action in the manual is:  
  - **“Leave the TV in standby with the mains power on for at least 60 minutes so the pixel refresh can complete.”** [E11]  

Operationally for your team:  
- Have the customer confirm the model is OLED O65 and the on‑screen code is **LX‑90**. [E6][E11]  
- Instruct them to keep the TV plugged into mains power, put it into **standby**, and leave it undisturbed for **at least 60 minutes** so the pixel refresh can complete. [E11]  
- If the problem persists after following that instruction, the manual’s guidance is to **contact customer care with the error code**, and it does **not** specify any further model‑specific repair steps for LX‑90. [E11][E12]  
- Users should **not open the TV or attempt internal repairs**; those are for qualified service personnel only. [E11][E12]  

---

3) Current OLED O65 tickets (Open / In Progress)  

- There are **4 current tickets** with product_model recorded as **"OLED O65"** and status **Open** or **In Progress**. [E7–E10]  
  - **2 Open:** TKT‑000120, TKT‑000017. [E7][E8]  
  - **2 In Progress:** TKT‑000114, TKT‑000080. [E9][E10]  

This count is based on the CRM product_model value “OLED O65,” which matches the OLED O65 model descriptions in those tickets. [E7–E10]

## The plan and the timeline
Overlapping bars = specialists running concurrently.

In [ ]:
for round_ in result.plan_history:
    print(f"Round {round_['round']} - {round_['trigger']}")
    print("  reasoning:", round_["reasoning"])
    for s in round_["steps"]:
        print(f"  {s['id']} -> {s['agent']}: {s['goal']}")
print()
print(result.timeline())

In [ ]:
# Each specialist's report: findings, confidence, evidence count
for r in result.reports:
    print(f"{r['step_id']} {r['agent']:<18} conf={r['confidence']:.2f} evidence={len(r['evidence'])} gaps={r['gaps'][:1]}")

In [ ]:
# Close the memory databases
await service.aclose()

## What to observe
- Three independent steps -> three parallel branches (no `depends_on`).
- The total time is close to the slowest specialist, not the sum.
- The final answer has one short section per domain, each with its own evidence IDs.